### Подготовка данных


In [1]:
import os
import shutil
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import joblib
import random

In [3]:
# Пути к папкам
home = '/home/slava/Documents/netology_ML/Diplom'
SCREENSHOTS_DIR = home + "/diff_spot_bunner_expected"
POPUPS_DIR = home + "/bunners_example"
OUTPUT_DIR = home + "/diff_spot_bunner_actual"

os.makedirs(OUTPUT_DIR, exist_ok=True)

popup_files = [f for f in os.listdir(POPUPS_DIR) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
if len(popup_files) == 0:
    raise ValueError("В папке с окнами нет изображений!")
popups = [Image.open(os.path.join(POPUPS_DIR, f)).convert("RGBA") for f in popup_files]

screenshot_files = [f for f in os.listdir(SCREENSHOTS_DIR) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
if len(screenshot_files) == 0:
    raise ValueError("В папке со скриншотами нет изображений!")

for s_file in screenshot_files:
    scr_path = os.path.join(SCREENSHOTS_DIR, s_file)
    scr_img = Image.open(scr_path).convert("RGBA")
    W_s, H_s = scr_img.size
    S_s = W_s * H_s

    popup = random.choice(popups)
    W_o, H_o = popup.size
    S_o = W_o * H_o

    area_ratio = random.uniform(0.01, 0.15)
    target_area = area_ratio * S_s

    scale = (target_area / S_o) ** 0.5
    W_new = int(W_o * scale)
    H_new = int(H_o * scale)

    if W_new > W_s or H_new > H_s:
        scale = min(W_s / W_o, H_s / H_o) * 0.99
        W_new = int(W_o * scale)
        H_new = int(H_o * scale)

    popup_resized = popup.resize((W_new, H_new), Image.Resampling.LANCZOS)

    x = random.randint(0, W_s - W_new)
    y = random.randint(0, H_s - H_new)

    scr_img.paste(popup_resized, (x, y), popup_resized)

    out_name = f"aug_{s_file}"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    scr_img.save(out_path, format="PNG")  # сохраняем в PNG для сохранения прозрачности

    print(f"Обработан {s_file} -> {out_name} (размер окна: {W_new}x{H_new}, позиция: ({x},{y}), площадь: {area_ratio*100:.1f}%)")

print("Готово!")

Обработан img_002807.jpg -> aug_img_002807.jpg (размер окна: 210x256, позиция: (497,132), площадь: 5.3%)
Обработан img_000660.jpg -> aug_img_000660.jpg (размер окна: 553x121, позиция: (208,110), площадь: 6.6%)
Обработан img_000653.jpg -> aug_img_000653.jpg (размер окна: 440x261, позиция: (726,49), площадь: 11.3%)
Обработан img_000426.jpg -> aug_img_000426.jpg (размер окна: 637x139, позиция: (588,11), площадь: 8.7%)
Обработан img_004377.jpg -> aug_img_004377.jpg (размер окна: 435x258, позиция: (473,235), площадь: 11.0%)
Обработан img_001972.jpg -> aug_img_001972.jpg (размер окна: 154x117, позиция: (898,224), площадь: 1.8%)
Обработан img_003769.jpg -> aug_img_003769.jpg (размер окна: 416x91, позиция: (563,672), площадь: 3.7%)
Обработан img_000334.jpg -> aug_img_000334.jpg (размер окна: 492x292, позиция: (96,105), площадь: 14.1%)
Обработан img_003595.jpg -> aug_img_003595.jpg (размер окна: 424x237, позиция: (549,395), площадь: 9.9%)
Обработан img_002140.jpg -> aug_img_002140.jpg (размер о